# Quantum + SVM Diabetes Prediction

This notebook downloads the diabetes dataset, preprocesses it, generates quantum features, trains an SVM classifier, and evaluates the results.

In [1]:
import os
import numpy as np
import pandas as pd
import kagglehub
import pennylane as qml

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)

## 1. Load dataset

In [2]:
dataset_path = kagglehub.dataset_download(
    "iammustafatz/diabetes-prediction-dataset"
)

print("Dataset downloaded to:")
print(dataset_path)
print("\nFiles:")
print(os.listdir(dataset_path))

csv_path = os.path.join(dataset_path, "diabetes_prediction_dataset.csv")
df = pd.read_csv(csv_path)

print("\nDataset shape:")
print(df.shape)
print("\nFirst 5 records:")
df.head()

Dataset downloaded to:
C:\Users\logit\.cache\kagglehub\datasets\iammustafatz\diabetes-prediction-dataset\versions\1

Files:
['diabetes_prediction_dataset.csv']

Dataset shape:
(100000, 9)

First 5 records:


,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level,diabetes
0,Female,80.0,0,1,never,25.19,6.6,140,0
1,Female,54.0,0,0,No Info,27.32,6.6,80,0
2,Male,28.0,0,0,never,27.32,5.7,158,0
3,Female,36.0,0,0,current,23.45,5.0,155,0
4,Male,76.0,1,1,current,20.14,4.8,155,0


## 2. Preprocessing

In [3]:
df = df.drop_duplicates()
df = pd.get_dummies(
    df,
    columns=["gender", "smoking_history"],
    drop_first=True,
)

X = df.drop("diabetes", axis=1).astype(float)
y = df["diabetes"]

print("Features:")
print(X.columns.tolist())

Features:
['age', 'hypertension', 'heart_disease', 'bmi', 'HbA1c_level', 'blood_glucose_level', 'gender_Male', 'gender_Other', 'smoking_history_current', 'smoking_history_ever', 'smoking_history_former', 'smoking_history_never', 'smoking_history_not current']


## 3. Train/test split and classical preprocessing

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

standard_scaler = StandardScaler()
X_train_scaled = standard_scaler.fit_transform(X_train)
X_test_scaled = standard_scaler.transform(X_test)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 76916
Testing samples: 19230


## 4. Feature selection and quantum encoding

In [5]:
n_qubits = 4

selector = SelectKBest(score_func=f_classif, k=n_qubits)
X_train_selected = selector.fit_transform(X_train_scaled, y_train)
X_test_selected = selector.transform(X_test_scaled)

selected_features = X.columns[selector.get_support()]
print("Selected features:")
for feature in selected_features:
    print(" -", feature)

quantum_scaler = MinMaxScaler(feature_range=(-np.pi, np.pi))
X_train_quantum_input = quantum_scaler.fit_transform(X_train_selected)
X_test_quantum_input = quantum_scaler.transform(X_test_selected)
print("Quantum input shape:", X_train_quantum_input.shape)

Selected features:
 - age
 - bmi
 - HbA1c_level
 - blood_glucose_level
Quantum input shape: (76916, 4)


## 5. Variational quantum circuit

In [6]:
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev)
def quantum_circuit(x, weights):
    for i in range(n_qubits):
        qml.RY(x[i], wires=i)

    for i in range(n_qubits):
        qml.RY(weights[i], wires=i)

    for i in range(n_qubits - 1):
        qml.CNOT(wires=[i, i + 1])

    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

np.random.seed(42)
weights = np.random.uniform(-np.pi, np.pi, n_qubits)
print("Initial quantum parameters:")
print(weights)

Initial quantum parameters:
[-0.78828768  2.83192151  1.45766093  0.61988954]


## 6. Generate quantum features

In [7]:
def quantum_feature_map(X):
    features = []
    for sample in X:
        features.append(quantum_circuit(sample, weights))
    return np.array(features)

X_train_q = quantum_feature_map(X_train_quantum_input)
X_test_q = quantum_feature_map(X_test_quantum_input)

print("Quantum feature shape:", X_train_q.shape)
print("Example quantum feature vector:")
print(X_train_q[0])

Quantum feature shape: (76916, 4)
Example quantum feature vector:
[0.92170606 0.69758335 0.47112051 0.16721962]


## 7. Train and evaluate the SVM

In [9]:
svm = SVC(kernel="rbf", probability=True, random_state=42)
svm.fit(X_train_q, y_train)

prediction = svm.predict(X_test_q)
probability = svm.predict_proba(X_test_q)[:, 1]

accuracy = accuracy_score(y_test, prediction)
auc = roc_auc_score(y_test, probability)

print(f"Quantum + SVM Accuracy: {accuracy * 100:.2f}%")
print(f"Quantum + SVM ROC-AUC: {auc:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, prediction))
print("Confusion Matrix:")
print(confusion_matrix(y_test, prediction))

c:\Users\logit\Downloads\prototype_ml\qml_env\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Quantum + SVM Accuracy: 92.11%
Quantum + SVM ROC-AUC: 0.7506

Classification Report:
              precision    recall  f1-score   support

           0       0.92      1.00      0.96     17534
           1       1.00      0.11      0.19      1696

    accuracy                           0.92     19230
   macro avg       0.96      0.55      0.57     19230
weighted avg       0.93      0.92      0.89     19230

Confusion Matrix:
[[17534     0]
 [ 1517   179]]


## 8. Test one patient

In [ ]:
def predict_patient(patient_data):
    patient_df = pd.DataFrame([patient_data])

    # Encode categorical features
    patient_df = pd.get_dummies(
        patient_df,
        columns=["gender", "smoking_history"],
        drop_first=True
    )

    # Make columns exactly match the training data
    patient_df = patient_df.reindex(
        columns=X.columns,
        fill_value=0
    )

    patient_df = patient_df.astype(float)

    # Same preprocessing used during training
    patient_scaled = standard_scaler.transform(patient_df)

    patient_selected = selector.transform(
        patient_scaled
    )

    patient_quantum = quantum_scaler.transform(
        patient_selected
    )

    # Quantum feature extraction
    quantum_features = quantum_feature_map(
        patient_quantum
    )

    # SVM prediction
    pred = svm.predict(
        quantum_features
    )[0]

    prob = svm.predict_proba(
        quantum_features
    )[0][1]

    return pred, prob


print("Quantum + SVM pipeline completed.")


# ============================================================
# TEST PATIENT
# ============================================================

patient = {
    "gender": "Female",
    "age": 45,
    "hypertension": 0,
    "heart_disease": 0,
    "smoking_history": "never",
    "bmi": 28.5,
    "HbA1c_level": 5.8,
    "blood_glucose_level": 120
}

prediction, probability = predict_patient(patient)

print("\nPatient Prediction:")
print("-------------------")

if prediction == 1:
    print("Prediction: Diabetes")
else:
    print("Prediction: No Diabetes")

print(
    f"Probability: {probability * 100:.2f}%"
)

IndentationError: unexpected indent (2462261625.py, line 32)